# 10 · Common Aug 5–21 comparison

Score saved v1 and v2 models on the **same** untouched holdout. This is the head-to-head quality table.

In [ ]:
import seaborn as sns
from matplotlib import pyplot as plt

from cross_model_drift.compare import compare_models, result_payload
from cross_model_drift.data import load_split
from cross_model_drift.models import load_model
from cross_model_drift.notebook import setup_model_session
from cross_model_drift.tracking import clearml_task, log_metrics

nb = setup_model_session()
holdout = load_split("v1", "holdout", nb.config, engine=nb.engine)
v1 = load_model(nb.artifacts / "models" / "v1_champion_challenger.joblib")
v2 = load_model(nb.artifacts / "models" / "v2_champion_challenger.joblib")
result = compare_models(holdout, v1, v2, threshold=nb.threshold)
result.quality

In [ ]:
result.predictions.to_parquet(nb.artifacts / "reports" / "holdout_predictions.parquet", index=False)
result.quality.to_csv(nb.artifacts / "reports" / "holdout_quality.csv", index=False)
payload = result_payload(result)
with clearml_task("compare_champion_challenger", config=nb.config, task_type="qc", tags=["compare"], init=False) as task:
    log_metrics(task, result.v1_metrics, title="v1")
    log_metrics(task, result.v2_metrics, title="v2")
payload

## Score distributions

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4.4))
sns.kdeplot(result.predictions["v1_score"], ax=ax, label="v1", color="#4C78A8")
sns.kdeplot(result.predictions["v2_score"], ax=ax, label="v2", color="#E45756")
ax.set_title("Holdout score distributions")
ax.set_xlabel("p(fraud)")
ax.legend()
nb.show(fig)